In [2]:
import requests, json

data = requests.get("https://rna.bgsu.edu/modified/modified_to_change_data.json").json()
print(f"{len(data)} modified components total")

u_children = {k: v for k, v in data.items() if v.get("standard_base") == ["U"]}
print(f"{len(u_children)} with parent U")

def glycosidic_swap(entry):
    swaps = {(c.get("parent_atom"), c.get("modified_atom"))
             for c in entry.get("changes", []) if c.get("change_type") == "replacement"}
    return ("N1", "C5") in swaps or ("C5", "N1") in swaps

fam = {k: v for k, v in u_children.items() if glycosidic_swap(v)}
for cid, e in sorted(fam.items(), key=lambda kv: -kv[1].get("count", 0)):
    mod = e.get("modomics", {}).get("short_name", "")
    print(f"{cid:5} n={e.get('count'):>6}  {mod:10} {e['pdb']['name'][:70]}")

874 modified components total
96 with parent U
PSU   n=  1118  pY         PSEUDOURIDINE-5'-MONOPHOSPHATE
3TD   n=   311  pm3Y       (1S)-1,4-anhydro-1-(3-methyl-2,4-dioxo-1,2,3,4-tetrahydropyrimidin-5-y
B8N   n=   150  B8N        (2~{R})-2-azanyl-4-[5-[(2~{S},3~{R},4~{S},5~{R})-3,4-bis(oxidanyl)-5-(
UY1   n=    92  pYm        2'-O-methylpseudouridine-5'-monophosphate
B8H   n=    39  pm1Y       [(2~{R},3~{S},4~{R},5~{S})-5-[1-methyl-2,4-bis(oxidanylidene)pyrimidin
C4J   n=    26  C4J        (5S)-5-{3-[(3S)-3-amino-3-carboxypropyl]-1-methyl-2,4-dioxo-1,2,3,4-te
FHU   n=     9  FHU        (5S,6R)-5-FLUORO-6-HYDROXY-PSEUDOURIDINE-5'-MONOPHOSPHATE
I2T   n=     7  pm1acp3Y   (2~{S})-2-azanyl-4-[5-[(2~{S},3~{R},4~{S},5~{R})-3,4-bis(oxidanyl)-5-(
M1Y   n=     1  M1Y        (1S)-1,4-anhydro-1-(1-methyl-2,4-dioxo-1,2,3,4-tetrahydropyrimidin-5-y


In [3]:
for cid in ["PUS", "PSU", "PSD", "PYU", "UPS"]:
    e = data.get(cid)
    print(cid, "—", e["pdb"]["name"] if e else "not in file", 
          "| parent:", e.get("standard_base") if e else "")

PUS — not in file | parent: 
PSU — PSEUDOURIDINE-5'-MONOPHOSPHATE | parent: ['U']
PSD — not in file | parent: 
PYU — not in file | parent: 
UPS — 2'-deoxy-5'-O-thiophosphonouridine | parent: ['DT']


In [4]:
import pandas as pd
df = pd.read_csv("psu_chains_meta.csv")
small = df[df.deposited_atoms < 12288].sort_values("deposited_atoms")
print(small[["pdb_id", "chain", "description", "resolution", "n_res_entity", "deposited_atoms"]].head(10))

    pdb_id chain                                        description  \
230   2Z9Q     A                                               tRNA   
162   2KYE     A  RNA (5'-R(*GP*AP*GP*AP*GP*(PSU)P*(PSU)P*GP*GP*...   
166   2GBH     A  5'-R(*(GMP)P*GP*GP*CP*UP*AP*AP*UP*GP*(PSU)P*UP...   
49    1BZT     A                                               TRNA   
48    1BZU     A                                               TRNA   
163   2KRZ     A  RNA (5'-R(*(PSU)P*CP*GP*GP*GP*CP*CP*CP*AP*UP*A...   
164   2KRY     A  RNA (5'-R(*(PSU)P*CP*GP*GP*GP*CP*CP*(RSQ)P*AP*...   
239   2LA9     A  RNA (5'-R(*GP*GP*GP*GP*AP*CP*UP*GP*UP*AP*AP*AP...   
173   2AWQ     A  5'-R(*GP*GP*GP*GP*AP*(PSU)P*UP*GP*AP*AP*AP*AP*...   
18    1LU3     A                         PHENYLALANINE TRANSFER RNA   

     resolution  n_res_entity  deposited_atoms  
230        11.7            75               75  
162         NaN            15              319  
166         NaN            24              337  
49          NaN       